# POC 01 — Bronze, Silver, and Gold
Connect `lh_sales` as the default Lakehouse before running the notebook.

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import (StructType, StructField, StringType,
    IntegerType, DecimalType, DateType)

customers_schema = StructType([
    StructField("customer_id", StringType(), False),
    StructField("customer_name", StringType(), True),
    StructField("city", StringType(), True),
    StructField("country", StringType(), True),
])
products_schema = StructType([
    StructField("product_id", StringType(), False),
    StructField("product_name", StringType(), True),
    StructField("category", StringType(), True),
    StructField("unit_price", DecimalType(12, 2), True),
])
orders_schema = StructType([
    StructField("order_id", StringType(), False),
    StructField("order_date", DateType(), True),
    StructField("customer_id", StringType(), True),
    StructField("product_id", StringType(), True),
    StructField("quantity", IntegerType(), True),
    StructField("status", StringType(), True),
])

In [ ]:
def read_csv(name, schema):
    return (spark.read
        .option("header", True)
        .schema(schema)
        .csv(f"Files/raw/{name}.csv")
        .withColumn("_ingested_at", F.current_timestamp())
        .withColumn("_source_file", F.input_file_name()))

bronze_customers = read_csv("customers", customers_schema)
bronze_products = read_csv("products", products_schema)
bronze_orders = read_csv("orders", orders_schema)

for name, frame in {
    "bronze_customers": bronze_customers,
    "bronze_products": bronze_products,
    "bronze_orders": bronze_orders,
}.items():
    frame.write.mode("overwrite").format("delta").saveAsTable(name)

In [ ]:
silver_customers = (bronze_customers
    .filter(F.col("customer_id").isNotNull())
    .dropDuplicates(["customer_id"]))

silver_products = (bronze_products
    .filter(F.col("product_id").isNotNull() & (F.col("unit_price") >= 0))
    .dropDuplicates(["product_id"]))

valid_order = (
    F.col("order_id").isNotNull()
    & F.col("order_date").isNotNull()
    & F.col("customer_id").isNotNull()
    & F.col("product_id").isNotNull()
    & (F.col("quantity") > 0)
)
rejected_orders = bronze_orders.filter(~valid_order)
silver_orders = bronze_orders.filter(valid_order).dropDuplicates(["order_id"])

for name, frame in {
    "silver_customers": silver_customers,
    "silver_products": silver_products,
    "silver_orders": silver_orders,
    "rejected_orders": rejected_orders,
}.items():
    frame.write.mode("overwrite").format("delta").saveAsTable(name)

print(f"Rejected orders: {rejected_orders.count()}")

In [ ]:
completed_sales = (silver_orders
    .filter(F.col("status") == "Completed")
    .join(silver_products, "product_id", "inner")
    .withColumn("revenue", F.col("quantity") * F.col("unit_price"))
    .withColumn("sale_month", F.trunc("order_date", "month")))

gold_sales_by_month = (completed_sales
    .groupBy("sale_month", "category")
    .agg(
        F.countDistinct("order_id").alias("orders_count"),
        F.sum("quantity").alias("units_sold"),
        F.sum("revenue").cast("decimal(14,2)").alias("revenue"),
    ))

gold_sales_by_month.write.mode("overwrite").format("delta").saveAsTable(
    "gold_sales_by_month"
)
display(gold_sales_by_month.orderBy("sale_month", "category"))

In [ ]:
assert silver_customers.count() == 3
assert silver_products.count() == 4
assert silver_orders.count() == 6
assert rejected_orders.count() == 0
assert gold_sales_by_month.count() == 4
total_revenue = gold_sales_by_month.agg(F.sum("revenue")).first()[0]
assert float(total_revenue) == 2170.00
print("Validations completed. Total revenue:", total_revenue)